# Biohub - Cell Tracking S1.5 Pipeline

このノートブックは、**S1.5フェーズ (スコアアップ戦略)** を実行するための統合検証ノートブックです。

In [ ]:
# === オフライン環境でのライブラリ自動セットアップ ===
import os
import sys
import glob

# 1. /kaggle/input 配下のすべての .whl ファイルから依存ライブラリ (zarr, tracksdata等) をインストール
wheels = glob.glob('/kaggle/input/**/*.whl', recursive=True)
if wheels:
    print(f'Found {len(wheels)} offline wheels. Installing...')
    wheel_dirs = set([os.path.dirname(w) for w in wheels])
    for w_dir in wheel_dirs:
        # zarr, tracksdata などのインストールを試みる
        !pip install --no-index --find-links={w_dir} zarr tracksdata
else:
    print('No offline wheels found in /kaggle/input.')

# 2. /kaggle/input から公式リポジトリのソースコード (kaggle_cell_tracking_competition/src) を自動探索してパスに追加
repo_candidates = glob.glob('/kaggle/input/**/kaggle_cell_tracking_competition/src', recursive=True)
if repo_candidates:
    repo_path = repo_candidates[0]
    print(f'Found repository path: {repo_path}')
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
else:
    print('Warning: kaggle_cell_tracking_competition/src was not found. Please ensure the source code dataset is added.')

In [ ]:
import os
import sys
import glob
import time
import numpy as np
import pandas as pd
import polars as pl
from skimage.feature import blob_dog

# パスの追加 (ローカル実行時のフォールバック)
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
SRC_DIR = os.path.join(BASE_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# kaggle_cell_tracking_competition/src のパス解決 (ローカルフォールバック)
repo_path_local = os.path.join(SRC_DIR, 'kaggle_cell_tracking_competition', 'src')
if repo_path_local not in sys.path:
    sys.path.insert(0, repo_path_local)

import tracksdata as td
from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import node_recall
from tracksdata.metrics import DistanceMatching

## 【ステップ 1 & 2】検出評価と検出器の向上

In [ ]:
def evaluate_detection_only(pred_nodes_df, gt_graph, scale, max_distance=7.0):
    """
    ステップ1: 検出単体の評価 (Recallの測定)
    """
    pred_graph = td.graph.InMemoryGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in pred_nodes_df.itertuples():
        pred_graph.add_node({
            't': int(row.t),
            'z': float(row.z),
            'y': float(row.y),
            'x': float(row.x)
        })
        
    matching = DistanceMatching(max_distance=max_distance, scale=scale)
    pred_graph.match(gt_graph, matching=matching)
    
    recall = node_recall(pred_graph, gt_graph)
    
    return {
        'node_recall': recall,
        'num_pred_nodes': pred_graph.num_nodes(),
        'num_gt_nodes': gt_graph.num_nodes()
    }

def detect_nodes_baseline(dataset_path, min_sigma=2.0, max_sigma=5.0, threshold=0.05):
    """
    ベースライン検出器 (blob_dog)
    """
    ds = open_dataset(dataset_path, normalize=True, require_tracks=False, device='cpu')
    images = ds.image
    n_frames = images.shape[0]
    
    nodes = []
    global_node_id = 0
    
    for t in range(n_frames):
        frame = images[t]
        if hasattr(frame, 'numpy'):
            frame = frame.numpy()
        
        img_min, img_max = frame.min(), frame.max()
        if img_max > img_min:
            img_norm = (frame.astype(np.float32) - img_min) / (img_max - img_min)
        else:
            img_norm = np.zeros_like(frame, dtype=np.float32)
            
        blobs = blob_dog(img_norm, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
        
        for blob in blobs:
            z, y, x, r = blob
            nodes.append({
                'node_id': global_node_id,
                't': t,
                'z': float(z),
                'y': float(y),
                'x': float(x)
            })
            global_node_id += 1
            
    return pd.DataFrame(nodes)

### 1.1 検出結果の3D視覚化 (Error Analysis)

予測された細胞位置 (ノード) と GT を3D空間にプロットし、公式マッチング基準 (7 µm) で正しく検出できたもの (TP: 緑)、余分な検出 (FP: 赤)、見落とした正解 (FN: 青) を色分けして可視化します。

In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

def plot_detection_3d(pred_nodes_df, gt_graph, t_selected=0, max_distance=7.0, scale=[1.0, 1.0, 1.0]):
    """
    指定したフレームにおける検出結果を3Dプロットし、TP/FP/FNを色分けします。
    """
    # GTの座標抽出
    gt_nodes_pl = gt_graph.node_attrs(attr_keys=['t', 'z', 'y', 'x'])
    gt_nodes_df = gt_nodes_pl.to_pandas()
    
    p_nodes = pred_nodes_df[pred_nodes_df['t'] == t_selected].copy()
    g_nodes = gt_nodes_df[gt_nodes_df['t'] == t_selected].copy()
    
    p_coords = p_nodes[['z', 'y', 'x']].values * scale
    g_coords = g_nodes[['z', 'y', 'x']].values * scale
    
    tp_p_idx, tp_g_idx = [], []
    if len(p_coords) > 0 and len(g_coords) > 0:
        tree = KDTree(g_coords)
        distances, indices = tree.query(p_coords, distance_upper_bound=max_distance)
        
        matched_g = set()
        for p_idx, (d, g_idx) in enumerate(zip(distances, indices)):
            if d <= max_distance and g_idx not in matched_g:
                tp_p_idx.append(p_idx)
                tp_g_idx.append(g_idx)
                matched_g.add(g_idx)
                
    tp_coords = p_coords[tp_p_idx] if tp_p_idx else np.empty((0, 3))
    fp_coords = np.delete(p_coords, tp_p_idx, axis=0) if len(p_coords) > 0 else np.empty((0, 3))
    fn_coords = np.delete(g_coords, tp_g_idx, axis=0) if len(g_coords) > 0 else np.empty((0, 3))
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    if len(tp_coords) > 0:
        ax.scatter(tp_coords[:, 2], tp_coords[:, 1], tp_coords[:, 0], c='green', label=f'TP (Correct Pred): {len(tp_coords)}', s=45, alpha=0.7)
    if len(fp_coords) > 0:
        ax.scatter(fp_coords[:, 2], fp_coords[:, 1], fp_coords[:, 0], c='red', label=f'FP (Extra Pred): {len(fp_coords)}', s=45, alpha=0.7)
    if len(fn_coords) > 0:
        ax.scatter(fn_coords[:, 2], fn_coords[:, 1], fn_coords[:, 0], c='blue', label=f'FN (Missing GT): {len(fn_coords)}', s=45, alpha=0.7)
        
    ax.set_title(f'3D Node Detection matching at Frame {t_selected}')
    ax.set_xlabel('X (um)')
    ax.set_ylabel('Y (um)')
    ax.set_zlabel('Z (um)')
    ax.legend()
    plt.show()

# 例: フレーム 0 で可視化を実行
if 'pred_nodes' in locals() and 'gt_graph' in locals():
    plot_detection_3d(pred_nodes, gt_graph, t_selected=0, scale=scale)

## 【ステップ 3 & 4】トラッキング評価とトラッカーの向上

In [ ]:
def evaluate_tracking_only(pred_edges, gt_graph, gt_nodes_df, scale):
    """
    ステップ3: トラッキング単体の評価 (GTノードを直接入力)
    """
    from tracking_cellmot.metrics import evaluate
    
    pred_graph = td.graph.InMemoryGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in gt_nodes_df.itertuples():
        pred_graph.add_node({
            'node_id': int(row.node_id),
            't': int(row.t),
            'z': float(row.z),
            'y': float(row.y),
            'x': float(row.x)
        })
        
    for edge in pred_edges:
        pred_graph.add_edge(int(edge['source_id']), int(edge['target_id']))
        
    res = evaluate(pred_graph, gt_graph, scale=scale)
    
    edge_denom = res.edge_tp + res.edge_fp + res.edge_fn
    edge_jaccard = res.edge_tp / edge_denom if edge_denom > 0 else 1.0
    
    return {
        'edge_jaccard': edge_jaccard,
        'edge_tp': res.edge_tp,
        'edge_fp': res.edge_fp,
        'edge_fn': res.edge_fn
    }

## 【ステップ 5 & 6】統合評価と間引き (Pruning) の実行

In [ ]:
# データ読み込みとベースラインのテスト実行
DATA_DIR = os.path.join(BASE_DIR, 'data', 'train')
zarr_paths = glob.glob(os.path.join(DATA_DIR, '*.zarr'))
if not zarr_paths:
    zarr_paths = glob.glob('/kaggle/input/**/train/*.zarr', recursive=True)

if zarr_paths:
    target_dataset_path = zarr_paths[0]
    print(f'Target dataset: {target_dataset_path}')
    
    # GTのロード
    ds_gt = open_dataset(target_dataset_path, normalize=True, require_tracks=True, device='cpu')
    gt_graph = ds_gt.tracks
    scale = ds_gt.scale
    
    # 1. 検出評価
    print('\n--- Running Node Detection Evaluation ---')
    pred_nodes = detect_nodes_baseline(target_dataset_path)
    det_res = evaluate_detection_only(pred_nodes, gt_graph, scale=scale)
    print(f'Node Recall: {det_res["node_recall"]:.4f}')
    
    # 可視化の実行
    print('\n--- Visualizing frame 0 cell centroids ---')
    plot_detection_3d(pred_nodes, gt_graph, t_selected=0, scale=scale)
    
    # 2. トラッキング評価 (GTノードを入力)
    print('\n--- Running Tracking Evaluation with GT Nodes ---')
    from track.btrack_tracker import run_btrack_tracking
    gt_nodes_pl = gt_graph.node_attrs(attr_keys=['node_id', 't', 'z', 'y', 'x'])
    gt_nodes_df = gt_nodes_pl.to_pandas()
    
    # btrack実行
    edges = run_btrack_tracking(gt_nodes_df, max_search_radius=25.0)
    track_res = evaluate_tracking_only(edges, gt_graph, gt_nodes_df, scale=scale)
    print(f'Edge Jaccard: {track_res["edge_jaccard"]:.4f} (TP={track_res["edge_tp"]}, FP={track_res["edge_fp"]}, FN={track_res["edge_fn"]})')